# Task 1: Netflix Data Cleaning and Preprocessing

**Objective:** Clean and prepare a raw dataset by handling missing values, duplicates, inconsistent formats, column names, dates, and data types.

**Deliverables:**
- Cleaned dataset: `cleaned_netflix_titles.csv`
- Short cleaning summary: `netflix_cleaning_summary.csv`

This notebook uses the Netflix Movies and TV Shows dataset from Kaggle: `netflix_titles.csv`.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 2. Load Dataset

The dataset used for this task is `netflix_titles.csv`.

In [ ]:
DATA_PATH = 'netflix_titles.csv'

df = pd.read_csv(DATA_PATH)

print('Original dataset shape:', df.shape)
df.head()

## 3. Initial Data Quality Check

In [ ]:
print('Dataset information:')
df.info()

missing_values = df.isnull().sum().sort_values(ascending=False)
missing_percent = (df.isnull().mean() * 100).round(2).sort_values(ascending=False)

missing_report = pd.DataFrame({
    'missing_values': missing_values,
    'missing_percent': missing_percent
})

print('\nNumber of duplicate rows:', df.duplicated().sum())
missing_report[missing_report['missing_values'] > 0]

## 4. Clean Column Names

This makes column names lowercase, removes extra spaces, and replaces symbols with underscores.

In [ ]:
df_clean = df.copy()

original_columns = df_clean.columns.tolist()

def clean_column_name(column):
    column = column.strip().lower()
    column = re.sub(r'[^a-z0-9]+', '_', column)
    column = re.sub(r'_+', '_', column)
    return column.strip('_')

df_clean.columns = [clean_column_name(col) for col in df_clean.columns]

column_name_changes = pd.DataFrame({
    'old_column_name': original_columns,
    'new_column_name': df_clean.columns
})

column_name_changes

## 5. Remove Duplicate Rows

In [ ]:
duplicates_before = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()
duplicates_after = df_clean.duplicated().sum()

print('Duplicate rows before cleaning:', duplicates_before)
print('Duplicate rows after cleaning:', duplicates_after)
print('Dataset shape after removing duplicates:', df_clean.shape)

## 6. Standardize Text Values

This removes extra spaces, changes blank strings to missing values, and standardizes Netflix categorical values such as `type` and `rating`.

In [ ]:
text_columns = df_clean.select_dtypes(include=['object', 'string']).columns

for col in text_columns:
    df_clean[col] = df_clean[col].astype('string').str.strip()
    df_clean[col] = df_clean[col].replace({'': pd.NA, 'nan': pd.NA, 'NaN': pd.NA, 'None': pd.NA})

# Some Netflix rows have duration values incorrectly stored in the rating column.
wrong_rating_mask = (
    df_clean['rating'].astype('string').str.contains(r'\b(?:min|season|seasons)\b', case=False, na=False)
    & df_clean['duration'].isna()
)
rating_duration_fixed = int(wrong_rating_mask.sum())
df_clean.loc[wrong_rating_mask, 'duration'] = df_clean.loc[wrong_rating_mask, 'rating']
df_clean.loc[wrong_rating_mask, 'rating'] = pd.NA

# Standardize type values while preserving the correct Netflix format.
type_map = {'movie': 'Movie', 'tv show': 'TV Show'}
df_clean['type'] = df_clean['type'].astype('string').str.lower().map(type_map).fillna(df_clean['type'])

# Keep ratings such as TV-MA, PG-13 and R in uppercase.
df_clean['rating'] = df_clean['rating'].astype('string').str.upper()

print('Rating-duration values fixed:', rating_duration_fixed)

df_clean.head()

## 7. Convert Date Columns

Columns with names containing `date`, `dt`, or `time` will be converted into datetime format. When exported, they will be saved in `dd-mm-yyyy` format.

In [ ]:
converted_date_columns = []

df_clean['date_added'] = pd.to_datetime(df_clean['date_added'], errors='coerce')
converted_date_columns.append('date_added')

print('Converted date columns:', converted_date_columns)

## 8. Fix Numeric Data Types

This converts `release_year` into a proper numeric column.

In [ ]:
converted_numeric_columns = []

df_clean['release_year'] = pd.to_numeric(df_clean['release_year'], errors='coerce').astype('Int64')
converted_numeric_columns.append('release_year')

print('Converted numeric columns:', converted_numeric_columns)
df_clean.dtypes

## 9. Handle Missing Values

Cleaning rule used here:
Netflix-specific cleaning rule used here:
- Fill missing `director`, `cast`, `country`, `rating`, and `duration` with `Unknown`.
- Fill missing `date_added` with the most common date in the dataset.
- Keep all rows and columns because the missing values are still manageable.

In [ ]:
missing_before = int(df_clean.isnull().sum().sum())

columns_to_drop = []

fill_values = {
    'director': 'Unknown',
    'cast': 'Unknown',
    'country': 'Unknown',
    'rating': 'Unknown',
    'duration': 'Unknown'
}

for col, value in fill_values.items():
    df_clean[col] = df_clean[col].fillna(value)

mode_date = df_clean['date_added'].mode(dropna=True)
if not mode_date.empty:
    df_clean['date_added'] = df_clean['date_added'].fillna(mode_date.iloc[0])

missing_after = int(df_clean.isnull().sum().sum())

print('Columns dropped:', columns_to_drop)
print('Total missing values before cleaning:', missing_before)
print('Total missing values after cleaning:', missing_after)

## 10. Optional: Check Outliers

Outliers are extreme values that may affect analysis. This cell detects them using the IQR method. Capping is turned off by default. If your task requires outlier treatment, change `CAP_OUTLIERS` to `True`.

In [ ]:
CAP_OUTLIERS = False

numeric_columns = df_clean.select_dtypes(include=[np.number]).columns
outlier_summary = []

for col in numeric_columns:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr
    outlier_count = ((df_clean[col] < lower_limit) | (df_clean[col] > upper_limit)).sum()
    
    outlier_summary.append({
        'column': col,
        'outlier_count': int(outlier_count),
        'lower_limit': lower_limit,
        'upper_limit': upper_limit
    })
    
    if CAP_OUTLIERS and iqr != 0:
        df_clean[col] = df_clean[col].clip(lower=lower_limit, upper=upper_limit)

outlier_report = pd.DataFrame(outlier_summary).sort_values('outlier_count', ascending=False)
outlier_report.head(10)

## 11. Final Quality Check

In [ ]:
print('Original dataset shape:', df.shape)
print('Cleaned dataset shape:', df_clean.shape)
print('Remaining duplicate rows:', df_clean.duplicated().sum())
print('Remaining missing values:', int(df_clean.isnull().sum().sum()))

df_clean.head()

## 12. Save Cleaned Dataset and Summary

In [ ]:
# Save date columns in dd-mm-yyyy format for the final CSV
df_export = df_clean.copy()

for col in df_export.columns:
    if pd.api.types.is_datetime64_any_dtype(df_export[col]):
        df_export[col] = df_export[col].dt.strftime('%d-%m-%Y')

cleaning_summary = pd.DataFrame({
    'item': [
        'original_rows',
        'original_columns',
        'cleaned_rows',
        'cleaned_columns',
        'duplicate_rows_removed',
        'missing_values_before',
        'missing_values_after',
        'columns_dropped',
        'rating_duration_values_fixed',
        'date_columns_converted',
        'numeric_columns_converted',
        'outlier_capping_applied'
    ],
    'result': [
        df.shape[0],
        df.shape[1],
        df_clean.shape[0],
        df_clean.shape[1],
        duplicates_before,
        missing_before,
        missing_after,
        ', '.join(columns_to_drop) if columns_to_drop else 'None',
        rating_duration_fixed,
        ', '.join(converted_date_columns) if converted_date_columns else 'None',
        ', '.join(converted_numeric_columns) if converted_numeric_columns else 'None',
        CAP_OUTLIERS
    ]
})

df_export.to_csv('cleaned_netflix_titles.csv', index=False)
cleaning_summary.to_csv('netflix_cleaning_summary.csv', index=False)

print('Saved cleaned_netflix_titles.csv')
print('Saved netflix_cleaning_summary.csv')

cleaning_summary

## 13. Short Summary of Changes

Use this summary in your GitHub README:

In this task, I cleaned and preprocessed the Netflix Movies and TV Shows dataset using Python Pandas. I checked the dataset for missing values, duplicate records, inconsistent text formats, incorrect data types, and date formatting issues. I renamed the column headers into a clean format, removed duplicate records if any, standardized text values, fixed rows where duration values were incorrectly placed under the rating column, converted `date_added` into datetime format, and converted `release_year` into a numeric column. Missing values in `director`, `cast`, `country`, `rating`, and `duration` were filled with `Unknown`, while missing dates were filled using the most common date. Finally, I exported the cleaned dataset as `cleaned_netflix_titles.csv` and created a short cleaning summary as `netflix_cleaning_summary.csv`.

## 14. Interview Questions and Short Answers

**1. What are missing values and how do you handle them?**  
Missing values are empty or unavailable data points. They can be handled by removing rows/columns, filling them with mean/median/mode, or using domain-based replacement.

**2. How do you treat duplicate records?**  
I check duplicates using `duplicated()` and remove them using `drop_duplicates()` if they are repeated records.

**3. Difference between `dropna()` and `fillna()` in Pandas?**  
`dropna()` removes missing values, while `fillna()` replaces missing values with another value such as mean, median, mode, or `Unknown`.

**4. What is outlier treatment and why is it important?**  
Outlier treatment means identifying and handling extreme values. It is important because outliers can affect averages, patterns, and model performance.

**5. Explain the process of standardizing data.**  
Standardizing data means making values consistent, such as changing `M`, `male`, and `Male` into one standard value: `Male`.

**6. How do you handle inconsistent data formats?**  
I convert the values into one format, such as changing all date values into a proper datetime type and exporting them as `dd-mm-yyyy`.

**7. What are common data cleaning challenges?**  
Common challenges include missing values, duplicates, inconsistent spelling, wrong data types, mixed date formats, and outliers.

**8. How can you check data quality?**  
I check data quality using missing value counts, duplicate checks, data type checks, unique value checks, summary statistics, and outlier detection.